In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate
from PIL import Image
import io

# Time and signal
t = np.linspace(0, 2, 400)
# signal = np.sin(2 * np.pi * 3 * t) * np.exp(-5*t) 
# signal = np.sin(2 * np.pi * 3 * t)
signal = np.exp(-3*t) 
lags = np.arange(-len(signal) + 1, len(signal))
acf = correlate(signal - np.mean(signal), signal - np.mean(signal), mode='full')
acf = acf / np.max(acf)

def create_frame(lag_idx):
    fig, axs = plt.subplots(2, 1, figsize=(8, 6), gridspec_kw={'height_ratios': [2, 1]})
    shift = lags[lag_idx]
    
    # Shifted version of signal
    shifted = np.zeros_like(signal)
    if shift < 0:
        shifted[:shift] = signal[-shift:]
    elif shift > 0:
        shifted[shift:] = signal[:-shift]
    else:
        shifted = signal

    # Overlap product
    product = signal * shifted

    # Plot signal and shifted version
    axs[0].plot(t, signal, label="Original", color='blue')
    axs[0].plot(t, shifted, label=f"Shifted (lag={shift})", color='orange', alpha=0.7)
    axs[0].fill_between(t, 0, product, color='green', alpha=0.3, label='Overlap (Product)')
    axs[0].legend()
    axs[0].set_ylim(-1.5, 1.5)
    axs[0].set_title("Sliding Signal for Autocorrelation")

    # Plot ACF as it builds
    axs[1].plot(lags[:lag_idx + 1], acf[:lag_idx + 1], color='purple')
    axs[1].set_xlim(lags[0], lags[-1])
    axs[1].set_ylim(-1.05, 1.05)
    axs[1].set_title("Autocorrelation Function Building Up")
    axs[1].set_xlabel("Lag")
    axs[1].set_ylabel("ACF")

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close()
    buf.seek(0)
    return Image.open(buf)

# Create GIF frames
frame_indices = range(100, 700, 1)  # adjust for smoother/faster gif
frames = [create_frame(i) for i in frame_indices]

# Save GIF
frames[0].save("acf_sliding_demo3.gif", save_all=True, append_images=frames[1:], duration=50, loop=None)
print("Saved GIF as 'acf_sliding_demo3.gif'")


Saved GIF as 'acf_sliding_demo3.gif'
